# Merge y conversión de Excel a JSON

Este notebook inspecciona los archivos en `data/informes_mensuales`, hace un merge por cada hoja (concatenando todas las hojas con el mismo nombre de todos los archivos Excel) y exporta los resultados a `data/json/`.

Estructura del notebook:
- Requisitos e imports
- Inspección: listar archivos y hojas
- Funciones: merge por hoja y export a JSON
- Ejecución (celda de prueba / smoke test)
- Notas sobre edge cases y cómo modificar

In [5]:
import json
from pathlib import Path
import pandas as pd
from typing import Dict, List

# Rutas base (ajusta si tu repo está en otra ubicación)
BASE_DIR = Path('.')  # asume que abres el notebook desde la raíz del repo
INFORMES_DIR = BASE_DIR / 'data' / 'informes_mensuales'
OUTPUT_DIR = BASE_DIR / 'data' / 'json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('informes dir ->', INFORMES_DIR.resolve())
print('output dir ->', OUTPUT_DIR.resolve())

informes dir -> C:\Users\DICA\Desktop\Repositorio\dashboard-app\notebooks\data\informes_mensuales
output dir -> C:\Users\DICA\Desktop\Repositorio\dashboard-app\notebooks\data\json


## Inspección: listar archivos Excel y sus hojas
La siguiente función lista todos los archivos `.xlsx` en `data/informes_mensuales` y devuelve sus hojas encontradas.

In [6]:
def list_excels_and_sheets(folder: Path) -> Dict[str, List[str]]:
    """Devuelve un diccionario {filename: [sheet_names]}
    No abre cada hoja, solo inspecciona los nombres de hoja para tener la estructura.
    """
    files = sorted(folder.glob('*.xlsx'))
    result = {}
    for p in files:
        try:
            xls = pd.ExcelFile(p)
            result[p.name] = xls.sheet_names
        except Exception as e:
            result[p.name] = f'ERROR: {e}'
    return result

# Ejemplo de uso: (ejecuta la celda para ver la salida)
# sheets = list_excels_and_sheets(INFORMES_DIR)
# from pprint import pprint
# pprint(sheets)

## Función principal: merge por hoja
Descripción:
- Detecta todos los nombres de hoja presentes en el conjunto de archivos.
- Para cada nombre de hoja, lee esa hoja en cada archivo (si existe), añade la columna `source_file` y las concatena con join externo (union de columnas).
- Devuelve un diccionario {sheet_name: DataFrame} listo para exportar.

In [3]:
def merge_excels_by_sheet(folder: Path) -> Dict[str, pd.DataFrame]:
    """Lee todos los excels en `folder` y concatena por hoja.
    Reglas:
      - union de columnas (outer join)
      - añade columna 'source_file' con el nombre del archivo original
    """
    files = sorted(folder.glob('*.xlsx'))
    # recolectar todos los nombres de hoja
    sheet_names = set()
    excel_objs = {}  # path -> pd.ExcelFile (cached)
    for p in files:
        try:
            xls = pd.ExcelFile(p)
            excel_objs[p] = xls
            sheet_names.update(xls.sheet_names)
        except Exception as e:
            print(f'WARNING: no se pudo leer {p.name}: {e}')
    sheet_names = sorted(sheet_names)
    merged = {}
    for sheet in sheet_names:
        frames = []
        for p, xls in excel_objs.items():
            if sheet in xls.sheet_names:
                try:
                    df = pd.read_excel(xls, sheet_name=sheet)
                    # Normalizar: si quieres, puedes limpiar nombres de columna aquí
                    df['source_file'] = p.name
                    frames.append(df)
                except Exception as e:
                    print(f'ERROR leyendo {p.name} -> sheet {sheet}: {e}')
        if frames:
            # concat con union de columnas (outer)
            merged_df = pd.concat(frames, ignore_index=True, sort=False)
            merged[sheet] = merged_df
        else:
            # no hay datos para esta hoja (raro)
            merged[sheet] = pd.DataFrame()
    return merged

# Nota: para hojas con formatos muy distintos puede que necesites más limpieza específica

## Exportar a JSON
Guardaremos un archivo JSON por cada hoja en `data/json/` y además un archivo `merged_all_sheets.json` que contiene un objeto con clave por hoja.

In [4]:
def export_dataframes_to_json(dfs: Dict[str, pd.DataFrame], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    summary = {}
    for sheet, df in dfs.items():
        safe_name = sheet.replace('/', '_').replace('\\', '_')
        out_file = output_dir / f'{safe_name}.json'
        # convertir a list of records para mejor compatibilidad
        try:
            records = json.loads(df.to_json(orient='records', force_ascii=False, date_format='iso'))
        except Exception:
            # fallback: convertir fila por fila
            records = df.to_dict(orient='records')
        with out_file.open('w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, indent=2)
        summary[sheet] = {'rows': len(df), 'file': str(out_file.relative_to(Path('.')))}
    # guardar el combinado
    combined = {sheet: json.loads(pd.Series(dfs[sheet].to_json(orient='records', force_ascii=False, date_format='iso')) if not dfs[sheet].empty else '[]') for sheet in dfs}
    combined_file = output_dir / 'merged_all_sheets.json'
    with combined_file.open('w', encoding='utf-8') as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)
    print('Export completado. Archivos escritos:')
    from pprint import pprint
    pprint(summary)

## Ejecución (smoke test)
Descomenta y ejecuta las siguientes celdas para listar hojas, hacer el merge y exportar a JSON.
Recomendación: ejecuta una celda a la vez para revisar resultados intermedios.

In [ ]:
# 1) Listar archivos y hojas
sheets = list_excels_and_sheets(INFORMES_DIR)
from pprint import pprint
pprint(sheets)

In [ ]:
# 2) Merge por hoja
merged = merge_excels_by_sheet(INFORMES_DIR)
# mostrar resumen por hoja
for sheet, df in merged.items():
    print(f